# FAISS Semantic Search Engine

This project implements a mini semantic search engine using Sentence Transformers and FAISS.

## Objectives

- Generate embeddings for a knowledge base
- Store embeddings in a FAISS vector index
- Perform semantic similarity search
- Retrieve the Top 3 most relevant results
- Build an interactive command-line search interface

# Task 1 — Setup and Embedding Generation

In this task, we create a small knowledge base containing common support-related questions and generate vector embeddings using the `all-MiniLM-L6-v2` Sentence Transformer model.

Each sentence will be converted into a 384-dimensional embedding vector.

In [3]:
from sentence_transformers import SentenceTransformer
import faiss
import numpy as np

print("All imports successful!")

All imports successful!


## Step 1: Create the Knowledge Base

We create a small knowledge base containing common customer support sentences related to password management, billing, account management, and login issues.

These sentences will later be converted into numerical vector representations called embeddings.

In [4]:
knowledge_base = [
    "You can reset your password by clicking on the forgot password link.",
    "Password reset instructions will be sent to your registered email address.",
    "You can update your billing information from the account settings page.",
    "Your monthly subscription payment can be viewed in the billing section.",
    "You can cancel your subscription at any time from your account settings.",
    "If you cannot log in, make sure you are using the correct email and password.",
    "Your account may be temporarily locked after multiple failed login attempts.",
    "You can change your registered email address from your profile settings.",
    "Contact customer support if you notice an unauthorized charge on your account.",
    "You can upgrade your subscription plan from the billing and plans section."
]

print(f"Total knowledge base sentences: {len(knowledge_base)}")

Total knowledge base sentences: 10


In [6]:
import os

SYSTEM_CA = "/etc/ssl/certs/ca-certificates.crt"

os.environ["REQUESTS_CA_BUNDLE"] = SYSTEM_CA
os.environ["SSL_CERT_FILE"] = SYSTEM_CA
os.environ["CURL_CA_BUNDLE"] = SYSTEM_CA

print("Using certificate bundle:", SYSTEM_CA)

Using certificate bundle: /etc/ssl/certs/ca-certificates.crt


In [7]:
model = SentenceTransformer("all-MiniLM-L6-v2")

print("Embedding model loaded successfully!")

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 1089.47it/s]


Embedding model loaded successfully!


In [8]:
embeddings = model.encode(knowledge_base)

print("Embedding matrix shape:", embeddings.shape)

Embedding matrix shape: (10, 384)


# Task 2 — Build the FAISS Index

In this task, we store the generated embeddings in a FAISS vector index.

The embeddings are normalized before being added to the index to enable cosine similarity behavior when using L2 distance.

We use `IndexFlatL2`, which performs exact nearest-neighbor search based on L2 distance.

In [9]:
# Convert embeddings to float32
embeddings = np.array(embeddings).astype("float32")

# Normalize embeddings
faiss.normalize_L2(embeddings)

# Create FAISS index
index = faiss.IndexFlatL2(384)

# Add embeddings to the index
index.add(embeddings)

# Print total vectors stored
print("Total vectors stored in FAISS:", index.ntotal)

Total vectors stored in FAISS: 10


# Task 3 — Semantic Search with FAISS

In this task, we convert a user query into an embedding using the same Sentence Transformer model.

The query embedding is normalized and searched against the FAISS index to retrieve the Top 3 most semantically similar knowledge base sentences.

In [10]:
# User query
query = "I forgot my password. How can I change it?"

# Generate query embedding
query_vector = model.encode([query])

# Convert to float32
query_vector = np.array(query_vector).astype("float32")

# Normalize query embedding
faiss.normalize_L2(query_vector)

# Search for Top 3 similar sentences
distances, indices = index.search(query_vector, k=3)

# Display results
print(f"Query: {query}\n")

print("Rank | Score | Matched Sentence")
print("-" * 80)

for rank, (score, idx) in enumerate(zip(distances[0], indices[0]), start=1):
    print(f"{rank} | {score:.4f} | {knowledge_base[idx]}")

Query: I forgot my password. How can I change it?

Rank | Score | Matched Sentence
--------------------------------------------------------------------------------
1 | 0.4176 | You can reset your password by clicking on the forgot password link.
2 | 0.6760 | Password reset instructions will be sent to your registered email address.
3 | 1.0700 | Contact customer support if you notice an unauthorized charge on your account.


## Step 4: Create a Reusable Semantic Search Function

To avoid repeating the embedding and FAISS search code for every query, we create a function that:

1. Converts the user query into an embedding.
2. Converts it to `float32`.
3. Normalizes the query vector.
4. Searches the FAISS index.
5. Displays the Top 3 matching sentences.

In [11]:
def semantic_search(query, k=3):
    # Generate embedding for the query
    query_vector = model.encode([query])

    # Convert to float32
    query_vector = np.array(query_vector).astype("float32")

    # Normalize the query vector
    faiss.normalize_L2(query_vector)

    # Search FAISS
    distances, indices = index.search(query_vector, k=k)

    # Display results
    print(f"\nQuery: {query}\n")
    print("Rank | Score | Matched Sentence")
    print("-" * 100)

    for rank, (score, idx) in enumerate(
        zip(distances[0], indices[0]), start=1
    ):
        print(f"{rank} | {score:.4f} | {knowledge_base[idx]}")

In [12]:
semantic_search("I need help resetting my password")

semantic_search("How can I update my payment details?")

semantic_search("Why am I unable to access my account?")


Query: I need help resetting my password

Rank | Score | Matched Sentence
----------------------------------------------------------------------------------------------------
1 | 0.5194 | You can reset your password by clicking on the forgot password link.
2 | 0.6235 | Password reset instructions will be sent to your registered email address.
3 | 1.0669 | Contact customer support if you notice an unauthorized charge on your account.

Query: How can I update my payment details?

Rank | Score | Matched Sentence
----------------------------------------------------------------------------------------------------
1 | 0.6024 | You can update your billing information from the account settings page.
2 | 1.0078 | You can upgrade your subscription plan from the billing and plans section.
3 | 1.0424 | Your monthly subscription payment can be viewed in the billing section.

Query: Why am I unable to access my account?

Rank | Score | Matched Sentence
----------------------------------------------

# Task 4 — Interactive CLI

In this task, we create a continuous command-line interface that allows users to enter search queries interactively.

For each query, the system retrieves and displays the Top 3 most relevant knowledge base sentences.

The user can type `exit` to stop the program.

In [14]:
while True:
    user_query = input("\nEnter your query (or type 'exit' to quit): ")

    if user_query.lower() == "exit":
        print("Exiting semantic search. Goodbye!")
        break

    semantic_search(user_query)


Query: How do I change my password?

Rank | Score | Matched Sentence
----------------------------------------------------------------------------------------------------
1 | 0.6781 | You can reset your password by clicking on the forgot password link.
2 | 0.8248 | Password reset instructions will be sent to your registered email address.
3 | 1.1192 | Contact customer support if you notice an unauthorized charge on your account.

Query: wxit

Rank | Score | Matched Sentence
----------------------------------------------------------------------------------------------------
1 | 1.8718 | Contact customer support if you notice an unauthorized charge on your account.
2 | 1.8821 | If you cannot log in, make sure you are using the correct email and password.
3 | 1.8879 | Your account may be temporarily locked after multiple failed login attempts.
Exiting semantic search. Goodbye!
